# Análise de lacunas da pesquisa em PLN na USP

Teses e dissertações da Biblioteca Digital da USP que tratam de Processamento de Linguagem Natural (PLN), coletadas com o Octoparse.

Etapas:
1. Leitura dos dados
2. Processamento dos resumos com o spaCy (tokenização, lematização e classes gramaticais)
3. Palavras e sintagmas mais frequentes
4. Palavras em alta e em queda ao longo do tempo
5. Ações de PLN e metodologia de cada trabalho
6. Modelagem de tópicos
7. Análise de lacunas
8. Resultados salvos

## 0. Preparação

> **Anotação:** esta etapa carrega as ferramentas usadas no notebook e define onde estão os dados. Nada é analisado ainda.
>
> - `pandas`: trabalha com tabelas (como uma planilha dentro do Python).
> - `matplotlib`, `seaborn` e `wordcloud`: fazem os gráficos e as nuvens de palavras.
> - `spacy`: biblioteca de PLN que entende a estrutura das frases em português.
> - `sklearn` (scikit-learn): usada na modelagem de tópicos.
>
> Se o modelo do spaCy ainda não estiver instalado, rode uma vez no terminal: `python -m spacy download pt_core_news_lg`

In [ ]:
import os
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print('Bibliotecas carregadas.')

In [ ]:
# Arquivo exportado do Octoparse. Se preferir CSV, troque o nome aqui (ex.: '../DATA/teses.csv')
ARQUIVO = '../DATA/Biblioteca_de_Teses_Scraping_xml.xlsx'

# Pasta onde os gráficos e as tabelas serão salvos
PASTA_RESULTADOS = '../RESULTADOS/'

# Quantos tópicos a modelagem deve encontrar (etapa 6)
NUMERO_DE_TOPICOS = 6

os.makedirs(PASTA_RESULTADOS, exist_ok=True)
print('Resultados serão salvos em', PASTA_RESULTADOS)

In [ ]:
print('Carregando o modelo de português do spaCy...')
nlp = spacy.load('pt_core_news_lg')
print('Modelo carregado.')

## 1. Leitura dos dados

> **Anotação:** lê a planilha do Octoparse e cria a tabela `teses`, com uma linha por trabalho. A única coluna nova é `ano`, tirada da data de defesa, porque as análises ao longo do tempo usam só o ano.
>
> Os gráficos desta etapa mostram o perfil da produção: quantos trabalhos foram defendidos por ano e em quais unidades da USP.

In [ ]:
print('Lendo o arquivo', ARQUIVO)

if ARQUIVO.endswith('.csv'):
    teses = pd.read_csv(ARQUIVO)
else:
    teses = pd.read_excel(ARQUIVO)

print(len(teses), 'trabalhos carregados.')
print('Colunas:', list(teses.columns))

# Cria a coluna 'ano' a partir da data de defesa
datas = pd.to_datetime(teses['data_defesa'])
teses['ano'] = datas.dt.year

# Sigla da unidade, para os gráficos ficarem legíveis: 'Escola Politécnica (EP)' vira 'EP'
siglas = []
for unidade in teses['unidade']:
    sigla = unidade.split('(')[-1]
    sigla = sigla.replace(')', '')
    siglas.append(sigla)
teses['sigla_unidade'] = siglas

print('Período:', teses['ano'].min(), 'a', teses['ano'].max())
teses.head()

In [ ]:
trabalhos_por_ano = teses['ano'].value_counts()
trabalhos_por_ano = trabalhos_por_ano.sort_index()

plt.figure(figsize=(12, 4))
plt.bar(trabalhos_por_ano.index, trabalhos_por_ano.values)
plt.title('Trabalhos defendidos por ano')
plt.xlabel('Ano de defesa')
plt.ylabel('Nº de trabalhos')
plt.savefig(PASTA_RESULTADOS + 'trabalhos_por_ano.png', bbox_inches='tight')
plt.show()

In [ ]:
trabalhos_por_unidade = teses['sigla_unidade'].value_counts()
trabalhos_por_unidade = trabalhos_por_unidade.sort_values()

plt.figure(figsize=(8, 5))
plt.barh(trabalhos_por_unidade.index, trabalhos_por_unidade.values)
plt.title('Trabalhos por unidade da USP')
plt.xlabel('Nº de trabalhos')
plt.savefig(PASTA_RESULTADOS + 'trabalhos_por_unidade.png', bbox_inches='tight')
plt.show()

## 2. Processamento dos resumos com o spaCy

> **Anotação:** é aqui que o PLN começa. O spaCy lê cada resumo e, para cada palavra, descobre:
>
> - **Token:** a palavra separada do resto do texto (tokenização).
> - **Lema:** a forma básica da palavra (*"classificadores"* vira *"classificador"*, *"avaliadas"* vira *"avaliar"*). Assim, variações da mesma palavra são contadas juntas.
> - **Classe gramatical (POS):** se é substantivo, verbo, adjetivo etc.
>
> O resultado de cada resumo é um "documento do spaCy", guardado na lista `resumos_processados`. Todas as etapas seguintes usam essa lista.

In [ ]:
print('Processando os resumos com o spaCy. Isso pode levar alguns minutos...')

resumos_processados = []
for resumo in teses['resumo']:
    documento = nlp(resumo)
    resumos_processados.append(documento)

print('Concluído:', len(resumos_processados), 'resumos processados.')

> **Anotação:** exemplo do que o spaCy produziu para as primeiras palavras do primeiro resumo.

In [ ]:
exemplo = []
for palavra in resumos_processados[0][:15]:
    exemplo.append([palavra.text, palavra.lemma_, palavra.pos_])

pd.DataFrame(exemplo, columns=['palavra', 'lema', 'classe gramatical'])

### Classes gramaticais nos resumos

> **Anotação:** conta quantas palavras de cada classe gramatical aparecem em todos os resumos. Mostra o perfil da escrita acadêmica (muitos substantivos e preposições, por exemplo) e justifica a próxima etapa, que mantém só substantivos, adjetivos e nomes próprios.

In [ ]:
NOMES_DAS_CLASSES = {
    'NOUN': 'substantivo', 'VERB': 'verbo', 'ADJ': 'adjetivo', 'ADV': 'advérbio',
    'PROPN': 'nome próprio', 'ADP': 'preposição', 'DET': 'artigo', 'PRON': 'pronome',
    'CCONJ': 'conjunção', 'AUX': 'verbo auxiliar', 'NUM': 'número',
}

contagem_classes = Counter()
for documento in resumos_processados:
    for palavra in documento:
        if palavra.pos_ in NOMES_DAS_CLASSES:
            nome = NOMES_DAS_CLASSES[palavra.pos_]
            contagem_classes[nome] += 1

classes = pd.Series(contagem_classes)
classes = classes.sort_values()

plt.figure(figsize=(8, 5))
plt.barh(classes.index, classes.values)
plt.title('Classes gramaticais nos resumos')
plt.xlabel('Nº de palavras')
plt.savefig(PASTA_RESULTADOS + 'classes_gramaticais.png', bbox_inches='tight')
plt.show()

## 3. Palavras e sintagmas mais frequentes

> **Anotação:** de cada resumo, ficam só as palavras que carregam conteúdo (substantivos, adjetivos e nomes próprios), já na forma de lema. Saem as *stopwords* (palavras muito comuns, como "de", "para", "que") e palavras que aparecem em quase todo resumo acadêmico e não dizem nada sobre o tema ("trabalho", "objetivo", "resultado"...).
>
> Também são extraídos os **sintagmas nominais**: grupos de palavras que formam um conceito, como *"reconhecimento de entidades nomeadas"*. Eles representam melhor os temas do que palavras soltas.
>
> A nuvem de palavras resume visualmente os termos mais frequentes: quanto maior a palavra, mais vezes ela aparece.

In [ ]:
# Palavras comuns em resumos acadêmicos que não indicam o tema do trabalho
STOPWORDS_ACADEMICAS = {
    'trabalho', 'estudo', 'objetivo', 'resultado', 'análise', 'método', 'pesquisa', 'dado',
    'abordagem', 'proposta', 'dissertação', 'tese', 'português', 'brasileiro', 'texto', 'linguagem',
}

CLASSES_DE_CONTEUDO = ['NOUN', 'ADJ', 'PROPN']


def palavras_importantes(documento):
    lista = []
    for palavra in documento:
        lema = palavra.lemma_.lower()
        if palavra.pos_ not in CLASSES_DE_CONTEUDO:
            continue
        if palavra.is_stop or not palavra.is_alpha or len(lema) <= 2:
            continue
        if lema in STOPWORDS_ACADEMICAS:
            continue
        lista.append(lema)
    return lista


def sintagmas(documento):
    lista = []
    for sintagma in documento.noun_chunks:
        palavras = []
        for palavra in sintagma:
            # tira o artigo do início ('o modelo de linguagem' vira 'modelo de linguagem')
            if palavra.i == sintagma.start and palavra.is_stop:
                continue
            palavras.append(palavra.text.lower())
        if len(palavras) >= 2:
            lista.append(' '.join(palavras))
    return lista


print('Extraindo palavras e sintagmas...')
todas_as_palavras = []
todos_os_sintagmas = []
for documento in resumos_processados:
    todas_as_palavras.append(palavras_importantes(documento))
    todos_os_sintagmas.append(sintagmas(documento))

teses['palavras'] = todas_as_palavras
teses['sintagmas'] = todos_os_sintagmas
print('Concluído.')

In [ ]:
contagem_palavras = Counter()
contagem_sintagmas = Counter()
for i in range(len(teses)):
    contagem_palavras.update(teses['palavras'][i])
    contagem_sintagmas.update(teses['sintagmas'][i])

print('Palavras diferentes:', len(contagem_palavras))
print('Sintagmas diferentes:', len(contagem_sintagmas))

palavras_frequentes = pd.Series(dict(contagem_palavras.most_common(20)))
sintagmas_frequentes = pd.Series(dict(contagem_sintagmas.most_common(20)))

fig, eixos = plt.subplots(1, 2, figsize=(15, 6))
eixos[0].barh(palavras_frequentes.index[::-1], palavras_frequentes.values[::-1])
eixos[0].set_title('Palavras mais frequentes')
eixos[1].barh(sintagmas_frequentes.index[::-1], sintagmas_frequentes.values[::-1])
eixos[1].set_title('Sintagmas mais frequentes')
plt.tight_layout()
plt.savefig(PASTA_RESULTADOS + 'palavras_e_sintagmas.png', bbox_inches='tight')
plt.show()

In [ ]:
nuvem = WordCloud(width=1200, height=600, background_color='white', colormap='viridis')
nuvem.generate_from_frequencies(contagem_palavras)

plt.figure(figsize=(14, 7))
plt.imshow(nuvem)
plt.axis('off')
plt.title('Nuvem de palavras dos resumos')
plt.savefig(PASTA_RESULTADOS + 'nuvem_de_palavras.png', bbox_inches='tight')
plt.show()

## 4. Palavras em alta e em queda

> **Anotação:** divide os trabalhos em dois períodos, antes e depois do ano central (a mediana dos anos de defesa), e compara a frequência de cada palavra nos dois. Palavras **em alta** passaram a aparecer mais; palavras **em queda**, menos. É a primeira pista de como a área mudou ao longo do tempo.
>
> A comparação usa a frequência **relativa** (a proporção da palavra no total de cada período), porque os dois períodos têm quantidades diferentes de texto.

In [ ]:
ano_central = int(teses['ano'].median())
print('Ano central:', ano_central)

contagem_antes = Counter()
contagem_depois = Counter()
for i in range(len(teses)):
    if teses['ano'][i] <= ano_central:
        contagem_antes.update(teses['palavras'][i])
    else:
        contagem_depois.update(teses['palavras'][i])

total_antes = sum(contagem_antes.values())
total_depois = sum(contagem_depois.values())

variacao = {}
for palavra in set(contagem_antes) | set(contagem_depois):
    frequencia_antes = contagem_antes[palavra] / total_antes
    frequencia_depois = contagem_depois[palavra] / total_depois
    variacao[palavra] = frequencia_depois - frequencia_antes

variacao = pd.Series(variacao)
variacao = variacao.sort_values()
em_queda = variacao.head(15)
em_alta = variacao.tail(15)

fig, eixos = plt.subplots(1, 2, figsize=(14, 6))
eixos[0].barh(em_alta.index, em_alta.values, color='seagreen')
eixos[0].set_title('Em alta depois de ' + str(ano_central))
eixos[1].barh(em_queda.index[::-1], em_queda.values[::-1], color='indianred')
eixos[1].set_title('Em queda depois de ' + str(ano_central))
plt.tight_layout()
plt.savefig(PASTA_RESULTADOS + 'palavras_em_alta_e_em_queda.png', bbox_inches='tight')
plt.show()

## 5. Ações de PLN e metodologia de cada trabalho

> **Anotação:** classifica cada trabalho com um **dicionário de expressões**. Se o título, as palavras-chave ou o resumo contêm, por exemplo, *"análise sintática"*, o trabalho recebe a ação **sintático**. Se contêm *"BERT"*, recebe a metodologia **transformers e BERT**.
>
> - **Ação de PLN:** em qual nível da arquitetura clássica do PLN o trabalho atua.
> - **Metodologia:** como o trabalho faz isso (regras, aprendizado de máquina, redes neurais, transformers, LLMs).
>
> O `PhraseMatcher` do spaCy procura as expressões ignorando maiúsculas e minúsculas. O dicionário tem termos em inglês porque muitos resumos em português usam termos técnicos em inglês. Um trabalho pode ter mais de uma ação e mais de uma metodologia.
>
> Para melhorar a classificação, leia os trabalhos que ficaram sem ação identificada e acrescente expressões ao dicionário.

### Níveis da arquitetura clássica do PLN

> **Anotação:** as ações seguem a divisão clássica do PLN em **níveis de análise linguística**, do som da fala até o contexto de uso (Liddy, 2001; Jurafsky; Martin, 2009; Indurkhya; Damerau, 2010), mais a **geração de linguagem**, a etapa que produz texto em vez de analisá-lo (Reiter; Dale, 2000; Gatt; Krahmer, 2018).
>
> | Nível | O que é analisado | Exemplos de ações |
> |---|---|---|
> | Fonético-fonológico | Sons da fala | reconhecimento e síntese de fala, prosódia |
> | Morfológico-lexical | Formação e classe das palavras | etiquetagem morfossintática, lematização, léxicos |
> | Sintático | Estrutura das frases | análise sintática, *treebanks*, gramáticas |
> | Semântico | Significado de palavras, frases e textos | desambiguação, entidades nomeadas, extração de informação, sentimentos, classificação e recuperação de textos |
> | Discursivo-pragmático | Relação entre frases, intenção e contexto | coesão, argumentação, diálogo, perguntas e respostas, ironia, discurso de ódio |
> | Geração de linguagem | Produção de texto a partir de dados ou de outro texto | tradução automática, sumarização, simplificação, geração de texto |
>
> Os níveis morfológico e lexical, e discursivo e pragmático, foram agrupados porque costumam aparecer juntos nos trabalhos e porque separá-los deixaria categorias com pouquíssimos casos.

In [ ]:
ACOES_PLN = {
    'fonético-fonológico': ['reconhecimento de fala', 'síntese de fala', 'speech recognition', 'text-to-speech',
                            'fala', 'prosódia', 'fonologia', 'fonético', 'fonema', 'acento'],
    'morfológico-lexical': ['etiquetagem', 'etiquetador', 'part-of-speech', 'classe gramatical',
                            'classes gramaticais', 'análise morfológica', 'analisador morfológico',
                            'análise morfossintática', 'morfossintática', 'morfologia', 'lematização',
                            'lematizador', 'stemming', 'tokenização', 'léxico', 'léxicos', 'construção de um léxico',
                            'terminologia', 'dicionário', 'dicionários'],
    'sintático': ['análise sintática', 'analisador sintático', 'parser', 'parsing', 'treebank', 'gramática',
                  'gramáticas', 'árvore sintática', 'dependências sintáticas', 'sintaxe'],
    'semântico': ['análise semântica', 'representação semântica', 'semântica lexical', 'papéis semânticos',
                  'desambiguação', 'implicação textual', 'similaridade semântica', 'ontologia', 'ontologias',
                  'entidades nomeadas', 'reconhecimento de entidades', 'named entity', 'extração de informação',
                  'extração de informações', 'análise de sentimento', 'análise de sentimentos', 'sentiment analysis',
                  'mineração de opinião', 'polaridade', 'classificação de textos', 'classificação de texto',
                  'classificação de documentos', 'text classification', 'classificação multirrótulo',
                  'recuperação de informação', 'recuperação de informações', 'information retrieval',
                  'busca semântica', 'sparql', 'modelagem de tópicos', 'modelo de tópicos', 'topic modeling'],
    'discursivo-pragmático': ['discurso', 'análise do discurso', 'coesão', 'coerência', 'anáfora', 'correferência',
                              'argumentação', 'argumentativo', 'diálogo', 'perguntas e respostas',
                              'question answering', 'chatbot', 'agente conversacional', 'ironia', 'sarcasmo',
                              'discurso de ódio', 'hate speech', 'posicionamento', 'desinformação', 'fake news'],
    'geração de linguagem': ['geração de texto', 'geração automática', 'geração de linguagem', 'text generation',
                             'tradução automática', 'tradutor automático', 'machine translation', 'sumarização',
                             'summarization', 'sumário automático', 'simplificação textual', 'simplificação de textos',
                             'paráfrase'],
}

METODOLOGIAS = {
    'regras e léxicos': ['baseado em regras', 'baseada em regras', 'rule-based', 'regras linguísticas',
                         'baseado em léxico', 'baseada em léxico', 'autômatos', 'lógica'],
    'estatística e aprendizado de máquina': ['aprendizado de máquina', 'machine learning', 'aprendizado computacional',
                                             'aprendizagem de máquina', 'svm', 'naive bayes', 'random forest',
                                             'regressão logística', 'crf', 'cadeias de markov', 'tf-idf'],
    'redes neurais e embeddings': ['rede neural', 'redes neurais', 'neural network', 'lstm', 'cnn', 'word2vec',
                                   'word embeddings', 'embeddings', 'aprendizado profundo', 'deep learning',
                                   'representação vetorial', 'representações vetoriais'],
    'transformers e BERT': ['transformer', 'transformers', 'bert', 'bertimbau', 'roberta', 'atenção neural',
                            'ajuste fino', 'fine-tuning'],
    'LLMs': ['llm', 'llms', 'grandes modelos de linguagem', 'large language model', 'gpt', 'chatgpt',
             'llama', 'prompt', 'prompts', 'zero-shot'],
}

print(len(ACOES_PLN), 'níveis de ações de PLN e', len(METODOLOGIAS), 'metodologias no dicionário.')

In [ ]:
def criar_buscador(dicionario):
    buscador = PhraseMatcher(nlp.vocab, attr='LOWER')
    for nome, expressoes in dicionario.items():
        padroes = []
        for expressao in expressoes:
            padroes.append(nlp.make_doc(expressao))
        buscador.add(nome, padroes)
    return buscador


def encontrar(buscador, documento):
    encontrados = set()
    for id_da_regra, inicio, fim in buscador(documento):
        nome = nlp.vocab.strings[id_da_regra]
        encontrados.add(nome)
    return sorted(encontrados)


buscador_acoes = criar_buscador(ACOES_PLN)
buscador_metodologia = criar_buscador(METODOLOGIAS)

print('Classificando os trabalhos...')
lista_acoes = []
lista_metodologia = []
for i in range(len(teses)):
    texto_completo = teses['titulo'][i] + '. ' + teses['palavras_chave'][i] + '. ' + teses['resumo'][i]
    documento = nlp.make_doc(texto_completo)
    lista_acoes.append(encontrar(buscador_acoes, documento))
    lista_metodologia.append(encontrar(buscador_metodologia, documento))

teses['acoes'] = lista_acoes
teses['metodologia'] = lista_metodologia

sem_acao = 0
sem_metodologia = 0
for i in range(len(teses)):
    if len(teses['acoes'][i]) == 0:
        sem_acao += 1
    if len(teses['metodologia'][i]) == 0:
        sem_metodologia += 1
print('Concluído.')
print('Trabalhos sem ação identificada:', sem_acao, 'de', len(teses))
print('Trabalhos sem metodologia identificada:', sem_metodologia, 'de', len(teses))

> **Anotação:** para contar e cruzar as categorias, cada combinação vira uma linha. Um trabalho com 2 ações e 1 metodologia gera 2 linhas na tabela `acoes_por_trabalho` e 2 linhas na tabela `acao_e_metodologia`.

In [ ]:
linhas_acoes = []
linhas_metodologia = []
linhas_cruzadas = []

for i in range(len(teses)):
    for acao in teses['acoes'][i]:
        linhas_acoes.append({'doi': teses['doi'][i], 'ano': teses['ano'][i],
                             'sigla_unidade': teses['sigla_unidade'][i], 'acao': acao})
        for metodologia in teses['metodologia'][i]:
            linhas_cruzadas.append({'acao': acao, 'metodologia': metodologia})
    for metodologia in teses['metodologia'][i]:
        linhas_metodologia.append({'ano': teses['ano'][i], 'metodologia': metodologia})

acoes_por_trabalho = pd.DataFrame(linhas_acoes)
metodologia_por_trabalho = pd.DataFrame(linhas_metodologia)
acao_e_metodologia = pd.DataFrame(linhas_cruzadas)
print('Tabelas montadas.')

In [ ]:
contagem_acoes = acoes_por_trabalho['acao'].value_counts()
# Ordem dos níveis da arquitetura (do som da fala até a geração), de cima para baixo no gráfico
contagem_acoes = contagem_acoes.reindex(list(ACOES_PLN)[::-1], fill_value=0)

contagem_metodologia = metodologia_por_trabalho['metodologia'].value_counts()
contagem_metodologia = contagem_metodologia.sort_values()

fig, eixos = plt.subplots(1, 2, figsize=(16, 6))
eixos[0].barh(contagem_acoes.index, contagem_acoes.values)
eixos[0].set_title('Ações de PLN por nível da arquitetura (nº de trabalhos)')
eixos[1].barh(contagem_metodologia.index, contagem_metodologia.values, color='darkorange')
eixos[1].set_title('Metodologia (nº de trabalhos)')
plt.tight_layout()
plt.savefig(PASTA_RESULTADOS + 'acoes_e_metodologia.png', bbox_inches='tight')
plt.show()

> **Anotação:** como a metodologia usada nos trabalhos mudou ao longo dos anos.

In [ ]:
metodologia_por_ano = pd.crosstab(metodologia_por_trabalho['ano'], metodologia_por_trabalho['metodologia'])

metodologia_por_ano.plot(kind='bar', stacked=True, figsize=(14, 5))
plt.title('Metodologia por ano de defesa')
plt.xlabel('Ano de defesa')
plt.ylabel('Nº de trabalhos')
plt.legend(title='Metodologia')
plt.savefig(PASTA_RESULTADOS + 'metodologia_por_ano.png', bbox_inches='tight')
plt.show()

## 6. Modelagem de tópicos

> **Anotação:** enquanto a etapa 5 usa categorias definidas por nós, a modelagem de tópicos deixa o próprio algoritmo **agrupar os trabalhos por assunto**, sem dicionário. Ela pode revelar temas que não estavam previstos.
>
> 1. **TF-IDF:** transforma cada resumo em números. Cada palavra recebe um peso maior quando é frequente naquele resumo e rara nos outros, ou seja, quando é característica dele.
> 2. **NMF:** encontra grupos de palavras que costumam aparecer juntas. Cada grupo é um tópico, e cada trabalho é associado ao tópico mais forte no seu resumo.
>
> O nome de cada tópico é formado pelas suas três palavras mais importantes. As nuvens de palavras mostram o conteúdo de cada tópico.

In [ ]:
textos_para_topicos = []
for palavras in teses['palavras']:
    textos_para_topicos.append(' '.join(palavras))

print('Calculando o TF-IDF...')
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)
matriz_tfidf = tfidf.fit_transform(textos_para_topicos)
vocabulario = tfidf.get_feature_names_out()
print('Vocabulário com', len(vocabulario), 'termos.')

print('Encontrando', NUMERO_DE_TOPICOS, 'tópicos com NMF...')
modelo_topicos = NMF(n_components=NUMERO_DE_TOPICOS, random_state=42, init='nndsvda', max_iter=500)
peso_dos_topicos = modelo_topicos.fit_transform(matriz_tfidf)
print('Concluído.')

In [ ]:
nomes_dos_topicos = []
termos_dos_topicos = []
for pesos in modelo_topicos.components_:
    posicoes = pesos.argsort()[::-1][:10]
    termos = []
    for posicao in posicoes:
        termos.append(vocabulario[posicao])
    termos_dos_topicos.append(termos)
    nomes_dos_topicos.append(', '.join(termos[:3]))

topico_de_cada_trabalho = []
for pesos in peso_dos_topicos:
    numero = pesos.argmax()
    topico_de_cada_trabalho.append(nomes_dos_topicos[numero])
teses['topico'] = topico_de_cada_trabalho

topicos = pd.DataFrame({'tópico': nomes_dos_topicos, 'termos principais': termos_dos_topicos})
trabalhos_por_topico = teses['topico'].value_counts()
topicos['nº de trabalhos'] = topicos['tópico'].map(trabalhos_por_topico)
topicos

In [ ]:
colunas = 3
linhas = (NUMERO_DE_TOPICOS + colunas - 1) // colunas
fig, eixos = plt.subplots(linhas, colunas, figsize=(18, 5 * linhas))
eixos = eixos.flatten()

for numero in range(NUMERO_DE_TOPICOS):
    pesos = modelo_topicos.components_[numero]
    frequencias = {}
    for posicao in pesos.argsort()[::-1][:40]:
        frequencias[vocabulario[posicao]] = pesos[posicao]
    nuvem = WordCloud(width=600, height=400, background_color='white', colormap='viridis')
    nuvem.generate_from_frequencies(frequencias)
    eixos[numero].imshow(nuvem)
    eixos[numero].set_title('Tópico: ' + nomes_dos_topicos[numero])
    eixos[numero].axis('off')

for numero in range(NUMERO_DE_TOPICOS, len(eixos)):
    eixos[numero].axis('off')

plt.tight_layout()
plt.savefig(PASTA_RESULTADOS + 'nuvens_dos_topicos.png', bbox_inches='tight')
plt.show()

> **Anotação:** mapa de calor mostrando em quais unidades da USP cada tópico aparece.

In [ ]:
topico_por_unidade = pd.crosstab(teses['topico'], teses['sigla_unidade'])

plt.figure(figsize=(14, 5))
sns.heatmap(topico_por_unidade, annot=True, fmt='d', cmap='Blues')
plt.title('Tópicos por unidade da USP')
plt.xlabel('Unidade')
plt.ylabel('Tópico')
plt.savefig(PASTA_RESULTADOS + 'topicos_por_unidade.png', bbox_inches='tight')
plt.show()

## 7. Análise de lacunas

> **Anotação:** cruza as ações de PLN com outras informações para mostrar **o que ainda foi pouco pesquisado**. Os mapas de calor funcionam como uma tabela colorida: quanto mais escura a célula, mais trabalhos; células com **0** são combinações que nenhum trabalho estudou.
>
> - **Ação × metodologia:** em quais níveis da arquitetura cada metodologia já foi usada.
> - **Ação × unidade:** em quais unidades cada nível aparece.
> - **Ação × década:** como o interesse por cada nível mudou ao longo do tempo.
>
> No fim, o código lista as lacunas sozinho, a partir dos próprios dados: níveis com menos trabalhos que a mediana e combinações de ação e metodologia sem nenhum trabalho.

In [ ]:
acao_x_metodologia = pd.crosstab(acao_e_metodologia['acao'], acao_e_metodologia['metodologia'])

# Inclui as ações e metodologias que não apareceram em nenhum trabalho (ficam com 0)
acao_x_metodologia = acao_x_metodologia.reindex(index=list(ACOES_PLN), columns=list(METODOLOGIAS), fill_value=0)

plt.figure(figsize=(10, 5))
sns.heatmap(acao_x_metodologia, annot=True, fmt='d', cmap='Blues')
plt.title('Ação de PLN × metodologia')
plt.xlabel('Metodologia')
plt.ylabel('Ação de PLN')
plt.savefig(PASTA_RESULTADOS + 'lacunas_acao_metodologia.png', bbox_inches='tight')
plt.show()

In [ ]:
acao_x_unidade = pd.crosstab(acoes_por_trabalho['acao'], acoes_por_trabalho['sigla_unidade'])
acao_x_unidade = acao_x_unidade.reindex(list(ACOES_PLN), fill_value=0)

plt.figure(figsize=(14, 5))
sns.heatmap(acao_x_unidade, annot=True, fmt='d', cmap='Blues')
plt.title('Ação de PLN × unidade da USP')
plt.xlabel('Unidade')
plt.ylabel('Ação de PLN')
plt.savefig(PASTA_RESULTADOS + 'lacunas_acao_unidade.png', bbox_inches='tight')
plt.show()

In [ ]:
acoes_por_trabalho['decada'] = (acoes_por_trabalho['ano'] // 10) * 10
acao_x_decada = pd.crosstab(acoes_por_trabalho['acao'], acoes_por_trabalho['decada'])
acao_x_decada = acao_x_decada.reindex(list(ACOES_PLN), fill_value=0)

plt.figure(figsize=(8, 5))
sns.heatmap(acao_x_decada, annot=True, fmt='d', cmap='Blues')
plt.title('Ação de PLN × década de defesa')
plt.xlabel('Década')
plt.ylabel('Ação de PLN')
plt.savefig(PASTA_RESULTADOS + 'lacunas_acao_decada.png', bbox_inches='tight')
plt.show()

In [ ]:
# Nº de trabalhos de cada ação, incluindo as que ficaram com 0
total_por_acao = contagem_acoes.reindex(list(ACOES_PLN), fill_value=0)
mediana = total_por_acao.median()

lacunas = []
for acao in ACOES_PLN:
    metodologias_nao_usadas = []
    for metodologia in METODOLOGIAS:
        if acao_x_metodologia.loc[acao, metodologia] == 0:
            metodologias_nao_usadas.append(metodologia)
    numero_de_trabalhos = int(total_por_acao[acao])
    lacunas.append({
        'ação de PLN': acao,
        'nº de trabalhos': numero_de_trabalhos,
        'abaixo da mediana': numero_de_trabalhos < mediana,
        'metodologias ainda não usadas': ', '.join(metodologias_nao_usadas),
    })

lacunas = pd.DataFrame(lacunas)
lacunas = lacunas.sort_values('nº de trabalhos')

print('Mediana de trabalhos por ação:', mediana)
print()
print('Ações com menos trabalhos que a mediana:')
for i in range(len(lacunas)):
    linha = lacunas.iloc[i]
    if linha['abaixo da mediana']:
        print('-', linha['ação de PLN'], '(' + str(linha['nº de trabalhos']) + ' trabalhos)')

lacunas

## 8. Resultados salvos

> **Anotação:** salva as tabelas finais na pasta `RESULTADOS`, junto com os gráficos gerados ao longo do notebook.
>
> - `trabalhos_classificados.csv`: cada trabalho com o que foi descoberto sobre ele (níveis de ação de PLN, metodologia e tópico).
> - `topicos.csv`: os tópicos encontrados e suas palavras principais.
> - `lacunas.csv`: a tabela de lacunas da etapa 7.

In [ ]:
trabalhos_classificados = pd.DataFrame({
    'doi': teses['doi'],
    'titulo': teses['titulo'],
    'ano': teses['ano'],
    'nivel': teses['nivel'],
    'unidade': teses['sigla_unidade'],
})

acoes_em_texto = []
metodologia_em_texto = []
for i in range(len(teses)):
    acoes_em_texto.append('; '.join(teses['acoes'][i]))
    metodologia_em_texto.append('; '.join(teses['metodologia'][i]))

trabalhos_classificados['acoes_pln'] = acoes_em_texto
trabalhos_classificados['metodologia'] = metodologia_em_texto
trabalhos_classificados['topico'] = teses['topico']

trabalhos_classificados.to_csv(PASTA_RESULTADOS + 'trabalhos_classificados.csv', index=False, encoding='utf-8')
topicos.to_csv(PASTA_RESULTADOS + 'topicos.csv', index=False, encoding='utf-8')
lacunas.to_csv(PASTA_RESULTADOS + 'lacunas.csv', index=False, encoding='utf-8')

print('Arquivos salvos em', PASTA_RESULTADOS)
for nome in sorted(os.listdir(PASTA_RESULTADOS)):
    print('-', nome)

## Referências da divisão em níveis

GATT, A.; KRAHMER, E. Survey of the state of the art in natural language generation: core tasks, applications and evaluation. **Journal of Artificial Intelligence Research**, v. 61, p. 65-170, 2018.

INDURKHYA, N.; DAMERAU, F. J. (ed.). **Handbook of Natural Language Processing**. 2. ed. Boca Raton: CRC Press, 2010.

JURAFSKY, D.; MARTIN, J. H. **Speech and Language Processing**. 2. ed. Upper Saddle River: Pearson Prentice Hall, 2009.

LIDDY, E. D. Natural language processing. In: **Encyclopedia of Library and Information Science**. 2. ed. New York: Marcel Dekker, 2001.

REITER, E.; DALE, R. **Building Natural Language Generation Systems**. Cambridge: Cambridge University Press, 2000.